# Imports

In [1]:
import os
import glob
import spacy
import itertools
import numpy as np
import pandas as pd
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

from sentence_transformers import SentenceTransformer
from bertopic.representation import PartOfSpeech, MaximalMarginalRelevance
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from gensim.utils import simple_preprocess

random_seed = 42
min_samples = 5
min_cluster_size = 15
data_path = '../../data/positives_260226.json'
output_dir = 'output/bertopic_260226'

/home/cs/grad/mazumdmm/Masud/YouTube Project/YouTube-4SE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.makedirs(output_dir, exist_ok=True)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model 'en_core_web_sm'...")
    from spacy.cli import download
    download("en_core_web_sm")

In [3]:
# Read all json files in the directory
df = pd.read_json(data_path)
print(f'Total videos: {len(df)}')

Total videos: 62760


In [4]:
# prepare documents for topic modeling (title + description)
ids = df['id']
documents = (df['title'] + "\n" + df['description'])
documents = [" ".join(doc.split()) for doc in documents]
print(f'Total documents: {len(documents)}')

Total documents: 62760


In [5]:
# Save documents to a text file
with open(os.path.join(output_dir, 'documents_260226.txt'), 'w', encoding='utf-8') as f:
    for doc in documents:
        f.write(doc + '\n')

In [9]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")
embeddings = embedding_model.encode(documents, normalize_embeddings=True, batch_size=2, show_progress_bar=True)
np.save(f'{output_dir}/embeddings_260226.npy', embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 415.21it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 31380/31380 [10:43<00:00, 48.79it/s] 


In [ ]:
# Apply BERTopic
topic_model = BERTopic(verbose=True)
topics, probs = topic_model.fit_transform(documents)

In [ ]:
# Display topics
topic_info = topic_model.get_topic_info()
topic_info

# V2

In [ ]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")
embeddings = embedding_model.encode(documents, normalize_embeddings=True, batch_size=2, show_progress_bar=True)
np.save(f'{output_dir}/embeddings.npy', embeddings)

In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=350, prediction_data=True)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
vectorizer_model = CountVectorizer(stop_words= "english", ngram_range=(1, 3), min_df=0.1)
representation_model = [PartOfSpeech("en_core_web_sm"), MaximalMarginalRelevance(top_n_words=20, diversity=0.3)]

In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=350, prediction_data=True)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
vectorizer_model = CountVectorizer(stop_words= "english", ngram_range=(1, 2), min_df=10)
representation_model = [PartOfSpeech("en_core_web_sm"), MaximalMarginalRelevance(diversity=0.3)]

topic_model = BERTopic(
    # embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    top_n_words=10,
    verbose=True,
)

In [36]:
topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)
topic_model.get_topic_info()

2026-03-10 15:56:01,197 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.01, n_components=5, n_jobs=1, random_state=42, verbose=True)
Tue Mar 10 15:56:01 2026 Construct fuzzy simplicial set
Tue Mar 10 15:56:01 2026 Finding Nearest Neighbors
Tue Mar 10 15:56:01 2026 Building RP forest with 16 trees
Tue Mar 10 15:56:02 2026 NN descent for 16 iterations
	 1  /  16
	 2  /  16
	 3  /  16
	 4  /  16
	Stopping threshold met -- exiting after 4 iterations
Tue Mar 10 15:56:07 2026 Finished Nearest Neighbor Search
Tue Mar 10 15:56:07 2026 Construct embedding


Epochs completed:   3%| ▎          6/200 [00:00]

	completed  0  /  200 epochs


Epochs completed:  12%| █▏         23/200 [00:02]

...

	completed  180  /  200 epochs


Epochs completed: 100%| ██████████ 200/200 [00:26]
2026-03-10 15:56:37,659 - BERTopic - Dimensionality - Completed ✓
2026-03-10 15:56:37,660 - BERTopic - Cluster - Start clustering the reduced embeddings


Tue Mar 10 15:56:37 2026 Finished embedding


2026-03-10 15:56:46,538 - BERTopic - Cluster - Completed ✓
2026-03-10 15:56:46,545 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-10 15:57:20,731 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,8165,-1_memory_clean code_clean_code,"[memory, clean code, clean, code, risk, stakeh...",[Episode 28: Colin Hammond: ScopeMaster and CO...
1,0,8272,0_testing_test_selenium_automation,"[testing, test, selenium, automation, tests, m...",[Selenium Tutorial For Beginners | What Is Sel...
2,1,6525,1_devops_azure_kubernetes_cloud,"[devops, azure, kubernetes, cloud, aws, docker...","[Containerization on AWS Explained | ECS, ECR,..."
3,2,3734,2_requirements_requirement_srs_business,"[requirements, requirement, srs, business, eli...",[Understanding the BABOK Guide Structure | Exc...
4,3,2989,3_sdlc_engineering_software_architecture,"[sdlc, engineering, software, architecture, ma...",[What Software Architecture Should Look Like W...
5,4,1934,4_ai_coding_agents_claude,"[ai, coding, agents, claude, copilot, agent, a...",[Cursor AI Agents Work Like 10 Developers (Cur...
6,5,1872,5_estimation_project_chart_scheduling,"[estimation, project, chart, scheduling, manag...",[software configuration management in urdu hin...
7,6,1755,6_security_cybersecurity_vulnerabilities_privacy,"[security, cybersecurity, vulnerabilities, pri...",[What Is A Good Process For Secure Code Review...
8,7,1680,7_scrum_sprint_agile_planning,"[scrum, sprint, agile, planning, methodology, ...",[SAFe Scaled Agile Framework 5.1 for Scrum Mas...
9,8,1666,8_uml_diagram_diagrams_sequence,"[uml, diagram, diagrams, sequence, case, class...",[Sequence Diagram Tutorial and EXAMPLE | UML D...


In [37]:
topic_info = topic_model.get_topic_info()
repr_docs = topic_model.get_representative_docs()

for topic_id in topic_info.Topic:
    if topic_id == -1:
        continue
    print(f"\nTopic {topic_id}: {topic_model.get_topic(topic_id)}")
    for doc in repr_docs[topic_id]:
        print("-", doc)


Topic 0: [('testing', np.float64(0.3377778515721071)), ('test', np.float64(0.3171498019676261)), ('selenium', np.float64(0.31164538504762157)), ('automation', np.float64(0.2721564339671332)), ('tests', np.float64(0.250231909332941)), ('manual', np.float64(0.24214043740342164)), ('quality', np.float64(0.23204208838196919)), ('manual testing', np.float64(0.22682406806786268)), ('unit', np.float64(0.22317627216832914)), ('tutorial', np.float64(0.22301741285063495))]
- Selenium Tutorial For Beginners | What Is Selenium? | Selenium Automation Testing Tutorial | Edureka 🔥 Edureka Selenium Training (Use Code "𝐘𝐎𝐔𝐓𝐔𝐁𝐄𝟐𝟎"): https://www.edureka.co/selenium-certification-training This Edureka Selenium tutorial video (Selenium Blog Series: https://goo.gl/VGBJHx) will give you an introduction to software testing. It talks about the drawbacks of manual testing and reasons why automation testing is the way forward. In this Selenium tutorial, you will also get to learn the different suites of Seleniu

In [48]:
from collections import defaultdict

topic_to_ids = defaultdict(list)

for doc_id, topic in zip(ids, topics):
    topic_to_ids[topic].append(doc_id)

In [49]:
topic_to_docs = defaultdict(list)

for doc_id, doc, topic in zip(ids, documents, topics):
    topic_to_docs[topic].append({
        "id": doc_id,
        "text": doc
    })

In [ ]:
df2 = pd.DataFrame({
    "id": ids,
    "document": documents,
    "topic": topics
})

In [56]:
df_sorted = df2.sort_values(by="topic")[["topic", "id", "document"]]

df_sorted.to_csv(f"{output_dir}/topic_documents.csv", index=False)

In [38]:
topic_model.save(f'{output_dir}/bt_48k_1003.model')
topic_model.save(f'{output_dir}/bt_48k_1003.pickle', serialization="pickle")

2026-03-10 15:57:44,531 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Tue Mar 10 15:57:45 2026 Building hub-based search tree
Tue Mar 10 15:57:45 2026 Forward diversification reduced edges from 730935 to 217455
Tue Mar 10 15:57:45 2026 Reverse diversification reduced edges from 217455 to 217455
Tue Mar 10 15:57:45 2026 Degree pruning reduced edges from 239100 to 239045
Tue Mar 10 15:57:45 2026 Resorting data and graph based on tree order
Tue Mar 10 15:57:45 2026 Building and compiling search function


2026-03-10 15:57:47,727 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


# Find Best Params

In [10]:
def compute_cv(topic_model, docs, top_n=10):
    texts = [simple_preprocess(doc) for doc in docs]
    dictionary = Dictionary(texts)
    
    topics = []
    for topic_id, words in topic_model.get_topics().items():
        if topic_id == -1:
            continue
        topics.append([w for w, _ in words[:top_n]])

    if len(topics) < 2:
        return None

    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v"
    )
    return cm.get_coherence()

In [13]:
def check(min_cluster_size, docs, embd):
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, prediction_data=True)
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
    vectorizer_model = CountVectorizer(stop_words= "english", ngram_range=(1, 3), min_df=10)
    representation_model = [PartOfSpeech("en_core_web_sm"), MaximalMarginalRelevance(diversity=0.3)]
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.01, metric='cosine', random_state=random_seed, verbose=True)

    model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        calculate_probabilities=False,
        top_n_words=20,
        verbose=False,
    )
    model.fit_transform(docs, embeddings=embd)
    score = compute_cv(model, docs)

    print("-"*50)
    print(f"Number of topics: {len(model.get_topics())}")
    print(f"Coherence score for {min_cluster_size}: {score}")
    print("-"*50)
    return score

In [14]:
sizes = [100, 150, 200, 250, 300, 350, 400, 450, 500]

for size in sizes:
    check(size, documents, embeddings)

UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.01, n_components=5, n_jobs=1, random_state=42, verbose=True)
Mon Mar 16 18:04:11 2026 Construct fuzzy simplicial set
Mon Mar 16 18:04:12 2026 Finding Nearest Neighbors
Mon Mar 16 18:04:12 2026 Building RP forest with 18 trees
Mon Mar 16 18:04:14 2026 NN descent for 16 iterations
	 1  /  16
	 2  /  16
	 3  /  16
	 4  /  16
	Stopping threshold met -- exiting after 4 iterations
Mon Mar 16 18:04:20 2026 Finished Nearest Neighbor Search
Mon Mar 16 18:04:20 2026 Construct embedding


Epochs completed:   2%| ▏          3/200 [00:00]

	completed  0  /  200 epochs


Epochs completed:  12%| █▏         23/200 [00:03]

	completed  20  /  200 epochs


...

	completed  160  /  200 epochs


Epochs completed:  92%| █████████▏ 183/200 [00:30]

	completed  180  /  200 epochs


Epochs completed: 100%| ██████████ 200/200 [00:33]


Mon Mar 16 18:25:30 2026 Finished embedding
--------------------------------------------------
Number of topics: 21
Coherence score for 500: 0.7670260216144841
--------------------------------------------------


# Final Topic Discovery

## Apply BERTopic

Best params:
- min_cluster_size: 350

In [5]:
embeddings = np.load(f"{output_dir}/embeddings.npy")

In [7]:
hdbscan_model = HDBSCAN(min_cluster_size=350, prediction_data=True)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
vectorizer_model = CountVectorizer(stop_words= "english", ngram_range=(1, 3), min_df=10)
representation_model = [PartOfSpeech("en_core_web_sm"), MaximalMarginalRelevance(diversity=0.3)]
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.01, metric='cosine', random_state=random_seed, verbose=True)

model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    calculate_probabilities=False,
    top_n_words=10,
    verbose=False,
)

In [8]:
topics, probs = model.fit_transform(documents, embeddings=embeddings)
model.get_topic_info()

UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.01, n_components=5, n_jobs=1, random_state=42, verbose=True)
Thu Mar 12 12:00:54 2026 Construct fuzzy simplicial set
Thu Mar 12 12:00:54 2026 Finding Nearest Neighbors
Thu Mar 12 12:00:54 2026 Building RP forest with 16 trees


/home/cs/grad/mazumdmm/Masud/YouTube Project/YouTube-4SE/.venv/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


Thu Mar 12 12:01:00 2026 NN descent for 16 iterations
	 1  /  16
	 2  /  16
	 3  /  16
	 4  /  16
	Stopping threshold met -- exiting after 4 iterations
Thu Mar 12 12:01:13 2026 Finished Nearest Neighbor Search
Thu Mar 12 12:01:15 2026 Construct embedding


Epochs completed:   2%| ▎          5/200 [00:00]

	completed  0  /  200 epochs


...

Epochs completed:  92%| █████████▏ 183/200 [00:24]

	completed  180  /  200 epochs


Epochs completed: 100%| ██████████ 200/200 [00:26]


Thu Mar 12 12:01:45 2026 Finished embedding


,Topic,Count,Name,Representation,Representative_Docs
0,-1,8165,-1_memory_clean code_clean_code,"[memory, clean code, clean, code, risk, handli...",[Episode 28: Colin Hammond: ScopeMaster and CO...
1,0,8272,0_testing_test_selenium_automation,"[testing, test, selenium, automation, tests, m...",[Selenium Tutorial For Beginners | What Is Sel...
2,1,6525,1_devops_azure_kubernetes_cloud,"[devops, azure, kubernetes, cloud, aws, docker...",[What Is A Build Server In Continuous Integrat...
3,2,3734,2_requirements_requirement_srs_business,"[requirements, requirement, srs, business, eli...",[Understanding the BABOK Guide Structure | Exc...
4,3,2989,3_sdlc_engineering_software_architecture,"[sdlc, engineering, software, architecture, ma...",[Introduction to Software Engineering Part 1 A...
5,4,1934,4_ai_coding_agents_claude,"[ai, coding, agents, claude, copilot, agent, a...",[Cursor AI Agents Work Like 10 Developers (Cur...
6,5,1872,5_estimation_project_chart_scheduling,"[estimation, project, chart, scheduling, manag...",[Software Project Management Week 10 | NPTEL A...
7,6,1755,6_security_cybersecurity_vulnerabilities_privacy,"[security, cybersecurity, vulnerabilities, pri...",[What Is A Good Process For Secure Code Review...
8,7,1680,7_scrum_sprint_agile_planning,"[scrum, sprint, agile, planning, methodology, ...",[SAFe Scaled Agile Framework 5.1 for Scrum Mas...
9,8,1666,8_uml_diagram_diagrams_sequence,"[uml, diagram, diagrams, sequence, case, class...",[Sequence Diagram Tutorial and EXAMPLE | UML D...


## Analyze

In [11]:
topic_info = model.get_topic_info()
repr_docs = model.get_representative_docs()

for topic_id in topic_info.Topic:
    if topic_id == -1:
        continue
    print(f"\nTopic {topic_id}: {model.get_topic(topic_id)}")
    for doc in repr_docs[topic_id]:
        print("-", doc)


Topic 0: [('testing', np.float64(0.33764887537480565)), ('test', np.float64(0.31588205778385303)), ('selenium', np.float64(0.30985742709073255)), ('automation', np.float64(0.27069511536123214)), ('tests', np.float64(0.24840719337361677)), ('manual', np.float64(0.24027138843119725)), ('quality', np.float64(0.23051227087694276)), ('manual testing', np.float64(0.22482787348826302)), ('tutorial', np.float64(0.22207798968127848)), ('unit', np.float64(0.22138717677362915))]
- Selenium Tutorial For Beginners | What Is Selenium? | Selenium Automation Testing Tutorial | Edureka 🔥 Edureka Selenium Training (Use Code "𝐘𝐎𝐔𝐓𝐔𝐁𝐄𝟐𝟎"): https://www.edureka.co/selenium-certification-training This Edureka Selenium tutorial video (Selenium Blog Series: https://goo.gl/VGBJHx) will give you an introduction to software testing. It talks about the drawbacks of manual testing and reasons why automation testing is the way forward. In this Selenium tutorial, you will also get to learn the different suites of Se

In [12]:
from collections import defaultdict

# Topic IDs
topic_to_ids = defaultdict(list)

for doc_id, topic in zip(ids, topics):
    topic_to_ids[topic].append(doc_id)

# Topic Docs
topic_to_docs = defaultdict(list)

for doc_id, doc, topic in zip(ids, documents, topics):
    topic_to_docs[topic].append({
        "id": doc_id,
        "text": doc
    })

df = pd.DataFrame({
    "id": ids,
    "document": documents,
    "topic": topics
})

In [13]:
df_sorted = df.sort_values(by="topic")[["topic", "id", "document"]]

df_sorted.to_csv(f"{output_dir}/topic_documents.csv", index=False)

In [ ]:
model.save(f'{output_dir}/bt_48k_1203.model')
model.save(f'{output_dir}/bt_48k_1203.pickle', serialization="pickle")